[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.5 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [19]:
# ✏️ YOUR IMPLEMENTATION HERE

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        d_head = d_model // num_heads
        self.d_head = d_head
        self.scale = 1 / math.sqrt(d_head)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )

    def _split_head(self, x):
        B, N, d_model = x.shape
        x = x.view(B, N, self.num_heads, self.d_head).transpose(1, 2)
        return x

    def _merge_heads(self, x):
        B, num_heads, N, d_head = x.shape
        x = x.transpose(1, 2).contiguous().view(B, N, num_heads * d_head)
        return x

    def _apply_causal_mask(self, x):
        B, num_heads, N, d_head = x.shape
        mask = torch.tril(
            torch.ones(N, N, dtype=torch.bool)
        )
        x = x.masked_fill(~mask, float('-inf'))
        return x

    def forward(self, x):
        B, N, d_model = x.shape
        # Pre-norm 1
        ln_x = self.ln1(x) # (B, N, d_model)
        # MHA
        q = self._split_head(self.W_q(ln_x))             # (B, num_heads, N, d_head)
        k = self._split_head(self.W_k(ln_x))             # (B, num_heads, N, d_head)
        v = self._split_head(self.W_v(ln_x))             # (B, num_heads, N, d_head)
        scores = q @ k.transpose(-2, -1) * self.scale
        masked_scores = self._apply_causal_mask(scores)
        weights = torch.softmax(masked_scores, dim=-1)
        attention = weights @ v
        attention = self._merge_heads(attention)      # (B, N, d_model)
        x = x + self.W_o(attention)
        # Pre-norm 2
        x = self.ln2(x)
        x = x + self.mlp(x)
        return x


In [20]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

Output shape: torch.Size([2, 8, 64])
Is nn.Module? True
Params: 49984


In [21]:
from torch_judge import check
check('gpt2_block')


🧪 Testing: GPT-2 Transformer Block (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (7.4ms)
  ✅ [2/5] Has LayerNorm (pre-norm architecture) (2.6ms)
  ✅ [3/5] MLP has 4x expansion with GELU (1.6ms)
  ✅ [4/5] Causal masking — future doesn't affect past (5.3ms)
  ✅ [5/5] Gradient flow to all parameters (4.9ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (21.8ms total)
  Progress saved. Run status() to see your dashboard.

